# CSCI4052 Group Project: WLASL Top-100 Build

This notebook creates a clean WLASL top-100 subset from the full dataset.

Outputs written to `wlasl_full_top_100_dataset/`:
- `dataset/`: Hugging Face dataset (Arrow)
- `class_statistics.csv` and `class_statistics.txt`: per-label video counts
- `labels_top100.json`: training/inference label metadata
- `id_to_gloss_top100.json`: numeric ID to ASL gloss mapping

In [1]:
from collections import Counter
from pathlib import Path
import json
import shutil
import pandas as pd
from datasets import ClassLabel, Video, load_dataset, load_from_disk

/mnt/linuxdata/CSCI4052-Course-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TOP_K = 100
DATASET_NAME = "Voxel51/WLASL"
SPLIT = "train"

# Prefer a previously downloaded full dataset if available.
LOCAL_FULL_DATASET_PATHS = [
    Path("wlasl_full_dataset/dataset"),
]

OUTPUT_DIR = Path("wlasl_full_top_100_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

id_to_gloss_candidates = [
    Path("wlasl_full_dataset/id_to_gloss.json"),
    Path("wlasl_full_top_100_dataset/id_to_gloss.json"),
]

id_to_gloss = {}
for candidate in id_to_gloss_candidates:
    if candidate.exists():
        with candidate.open("r", encoding="utf-8") as f:
            id_to_gloss = {str(k): str(v).strip().lower() for k, v in json.load(f).items()}
        print(f"Loaded id->gloss map from: {candidate}")
        break

local_path = next((p for p in LOCAL_FULL_DATASET_PATHS if p.exists()), None)
if local_path is not None:
    dataset = load_from_disk(str(local_path))
    print(f"Loaded local dataset from: {local_path} ({len(dataset)} samples)")
else:
    dataset = load_dataset(DATASET_NAME, split=SPLIT)
    print(f"Loaded {DATASET_NAME} [{SPLIT}] from Hugging Face ({len(dataset)} samples)")

if "video" in dataset.features:
    dataset = dataset.cast_column("video", Video(decode=False))

label_feature = dataset.features.get("label")
has_class_names = hasattr(label_feature, "names") and isinstance(label_feature.names, list)
print(f"Has class-name metadata: {has_class_names}")


def class_name_for_label(label_idx: int) -> str:
    if has_class_names:
        return label_feature.int2str(int(label_idx))
    return str(label_idx)


def raw_id_from_class_name(class_name: str) -> str:
    if class_name.startswith("data_"):
        suffix = class_name.split("data_")[-1]
        if suffix.isdigit():
            return suffix
    return class_name


def gloss_for_label(label_idx: int) -> str:
    class_name = class_name_for_label(label_idx)
    raw_id = raw_id_from_class_name(class_name)
    return id_to_gloss.get(str(raw_id), class_name).strip().lower()

Loaded id->gloss map from: wlasl_full_top_100_dataset/id_to_gloss.json


Rate limited. Waiting 197.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_40/23548.mp4
Rate limited. Waiting 197.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_40/23568.mp4
Rate limited. Waiting 197.0s before retry [Retry 1/5].
Rate limited. Waiting 197.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_40/23572.mp4
Rate limited. Waiting 197.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_40/23569.mp4
Rate limited. Waiting 197.0s before retry [Retry 1/5].
HTTP Error 429 thrown 

Loaded Voxel51/WLASL [train] from Hugging Face (11880 samples)
Has class-name metadata: True


In [3]:
label_list = [int(x) for x in dataset["label"]]
class_counts = Counter(label_list)

sorted_counts = sorted(
    class_counts.items(),
    key=lambda kv: (-kv[1], class_name_for_label(kv[0])),
)

top_k_items = sorted_counts[:TOP_K]
top_k_label_ids = [label_id for label_id, _ in top_k_items]
top_k_label_set = set(top_k_label_ids)

stats_rows = []
for new_idx, (label_id, count) in enumerate(top_k_items):
    class_name = class_name_for_label(label_id)
    gloss = gloss_for_label(label_id)
    stats_rows.append(
        {
            "new_label_idx": new_idx,
            "original_label_idx": int(label_id),
            "class_name": class_name,
            "gloss": gloss,
            "videos": int(count),
        }
    )

stats_df = pd.DataFrame(stats_rows)

print(f"Total samples in source split: {len(dataset)}")
print(f"Unique classes in source split: {len(class_counts)}")
print(f"Selected classes: {len(top_k_items)}")
print(f"Top-100 sample count (before remap): {sum(row['videos'] for row in stats_rows)}")

stats_df.head(15)

Total samples in source split: 11880
Unique classes in source split: 119
Selected classes: 100
Top-100 sample count (before remap): 10000


,new_label_idx,original_label_idx,class_name,gloss,videos
0,0,0,data_0,book,100
1,1,1,data_1,drink,100
2,2,2,data_10,deaf,100
3,3,3,data_100,visit,100
4,4,4,data_101,wait,100
5,5,5,data_102,water,100
6,6,6,data_103,wife,100
7,7,7,data_104,yellow,100
8,8,8,data_105,backpack,100
9,9,9,data_106,bar,100


In [4]:
# Filter dataset to top-100 classes only.
filtered_dataset = dataset.filter(lambda sample: int(sample["label"]) in top_k_label_set)

# Remap original label IDs -> [0..99] for clean training labels.
old_to_new = {old_id: new_id for new_id, old_id in enumerate(top_k_label_ids)}
new_class_names = [class_name_for_label(old_id) for old_id in top_k_label_ids]


def remap_label(sample):
    old_label = int(sample["label"])
    sample["orig_label"] = old_label
    sample["label"] = int(old_to_new[old_label])
    return sample

filtered_dataset = filtered_dataset.map(remap_label)
new_features = filtered_dataset.features.copy()
new_features["label"] = ClassLabel(names=new_class_names)
filtered_dataset = filtered_dataset.cast(new_features)

# Save Arrow dataset and metadata files.
dataset_out = OUTPUT_DIR / "dataset"
temp_out = OUTPUT_DIR / "dataset_tmp_write"

if temp_out.exists():
    shutil.rmtree(temp_out)

filtered_dataset.save_to_disk(str(temp_out))

if dataset_out.exists():
    shutil.rmtree(dataset_out)

temp_out.rename(dataset_out)

stats_df.to_csv(OUTPUT_DIR / "class_statistics.csv", index=False)
(OUTPUT_DIR / "class_statistics.txt").write_text(
    "\n".join([f"{row['gloss']}: {row['videos']}" for row in stats_rows]),
    encoding="utf-8",
)

labels_payload = {
    "num_classes": TOP_K,
    "labels": [row["gloss"] for row in stats_rows],
    "class_names": [row["class_name"] for row in stats_rows],
    "idx_to_gloss": {str(row["new_label_idx"]): row["gloss"] for row in stats_rows},
    "idx_to_class_name": {str(row["new_label_idx"]): row["class_name"] for row in stats_rows},
    "old_label_to_new_label": {str(old): int(new) for old, new in old_to_new.items()},
}

with (OUTPUT_DIR / "labels_top100.json").open("w", encoding="utf-8") as f:
    json.dump(labels_payload, f, indent=2)

id_to_gloss_top100 = {
    str(row["original_label_idx"]): row["gloss"] for row in stats_rows
}
with (OUTPUT_DIR / "id_to_gloss_top100.json").open("w", encoding="utf-8") as f:
    json.dump(id_to_gloss_top100, f, indent=2)

print(f"Saved top-100 dataset and metadata to: {OUTPUT_DIR}")
print(f"Filtered samples: {len(filtered_dataset)}")

Saving the dataset (9/9 shards): 100%|██████████| 10000/10000 [00:08<00:00, 1123.89 examples/s]

Saved top-100 dataset and metadata to: wlasl_full_top_100_dataset
Filtered samples: 10000


In [5]:
# Sanity checks
reloaded = load_from_disk(str(OUTPUT_DIR / "dataset"))
print(f"Reloaded dataset size: {len(reloaded)}")
print(f"Reloaded features: {reloaded.features}")

with (OUTPUT_DIR / "labels_top100.json").open("r", encoding="utf-8") as f:
    labels_check = json.load(f)

print(f"Saved labels: {len(labels_check['labels'])}")
print("First 10 labels:", labels_check["labels"][:10])
print("\nTop-20 class statistics preview:")
print(stats_df[["new_label_idx", "gloss", "videos"]].head(20).to_string(index=False))

Reloaded dataset size: 10000
Reloaded features: {'video': Video(decode=False, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact'), 'label': ClassLabel(names=['data_0', 'data_1', 'data_10', 'data_100', 'data_101', 'data_102', 'data_103', 'data_104', 'data_105', 'data_106', 'data_107', 'data_108', 'data_109', 'data_11', 'data_110', 'data_111', 'data_112', 'data_113', 'data_114', 'data_115', 'data_116', 'data_117', 'data_118', 'data_12', 'data_13', 'data_14', 'data_15', 'data_16', 'data_17', 'data_19', 'data_2', 'data_20', 'data_21', 'data_22', 'data_23', 'data_24', 'data_25', 'data_26', 'data_27', 'data_28', 'data_29', 'data_3', 'data_30', 'data_31', 'data_32', 'data_33', 'data_34', 'data_35', 'data_36', 'data_37', 'data_38', 'data_39', 'data_4', 'data_40', 'data_41', 'data_42', 'data_43', 'data_44', 'data_45', 'data_46', 'data_47', 'data_48', 'data_49', 'data_5', 'data_50', 'data_51', 'data_52', 'data_53', 'data_54', 'data_55', 'data_56', 'd